In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet(
    "../data/processed/causal_corrected_dataset.parquet"
)

print(df.shape)

(2112500, 34)


In [2]:
psm_df = df[
    [
        "sales",
        "discount_flag",
        "month",
        "week_of_year",
        "is_weekend",
        "has_event",
        "snap_CA",
        "snap_TX",
        "snap_WI"
    ]
].copy()

print(psm_df.shape)
psm_df.head()

(2112500, 9)


,sales,discount_flag,month,week_of_year,is_weekend,has_event,snap_CA,snap_TX,snap_WI
0,1,0,1,1,0,1,1,1,0
1,1,0,1,1,0,0,1,0,1
2,0,0,1,1,0,0,1,1,1
3,0,0,1,1,1,0,1,0,0
4,0,0,1,1,1,0,1,1,1


In [3]:
print("Treatment Distribution:")
print(psm_df["discount_flag"].value_counts())

print("\nTreatment Rate:")
print(psm_df["discount_flag"].mean())

Treatment Distribution:
discount_flag
0    2106990
1       5510
Name: count, dtype: int64

Treatment Rate:
0.002608284023668639


In [4]:
from sklearn.linear_model import LogisticRegression

X = psm_df[
    [
        "month",
        "week_of_year",
        "is_weekend",
        "has_event",
        "snap_CA",
        "snap_TX",
        "snap_WI"
    ]
]

y = psm_df["discount_flag"]

propensity_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

propensity_model.fit(X, y)

print("Propensity model trained successfully")

Propensity model trained successfully


In [5]:
psm_df["propensity_score"] = propensity_model.predict_proba(X)[:, 1]

print(
    psm_df["propensity_score"].describe()
)

count    2.112500e+06
mean     2.610408e-03
std      3.493855e-04
min      1.512032e-03
25%      2.356005e-03
50%      2.618514e-03
75%      2.867950e-03
max      3.325175e-03
Name: propensity_score, dtype: float64


In [6]:
from sklearn.neighbors import NearestNeighbors

treated = psm_df[
    psm_df["discount_flag"] == 1
].copy()

control = psm_df[
    psm_df["discount_flag"] == 0
].copy()

print("Treated:", len(treated))
print("Control:", len(control))

Treated: 5510
Control: 2106990


In [7]:
nn = NearestNeighbors(
    n_neighbors=1
)

nn.fit(
    control[["propensity_score"]]
)

distances, indices = nn.kneighbors(
    treated[["propensity_score"]]
)

print("Matching completed")

Matching completed


In [8]:
matched_controls = control.iloc[
    indices.flatten()
].copy()

print(
    matched_controls.shape
)

(5510, 10)


In [9]:
treated_sales = treated["sales"].mean()

control_sales = matched_controls["sales"].mean()

psm_ate = treated_sales - control_sales

print("Treated Mean Sales :", treated_sales)
print("Matched Control Mean Sales :", control_sales)
print("PSM Treatment Effect :", psm_ate)

Treated Mean Sales : 0.7012704174228676
Matched Control Mean Sales : 0.4847549909255898
PSM Treatment Effect : 0.21651542649727773


In [10]:
psm_results = {
    "treated_mean_sales": float(treated_sales),
    "matched_control_mean_sales": float(control_sales),
    "psm_ate": float(psm_ate),
    "dowhy_ate": -0.055745695026705455
}

import json

with open(
    "../outputs/psm_results.json",
    "w"
) as f:
    json.dump(
        psm_results,
        f,
        indent=4
    )

print("PSM results saved")

PSM results saved


In [11]:
counterfactual_df = pd.DataFrame({
    "observed_sales": treated["sales"].values,
    "counterfactual_sales": matched_controls["sales"].values
})

counterfactual_df["estimated_uplift"] = (
    counterfactual_df["observed_sales"]
    - counterfactual_df["counterfactual_sales"]
)

counterfactual_df.head()

,observed_sales,counterfactual_sales,estimated_uplift
0,0,1,-1
1,0,1,-1
2,0,0,0
3,2,0,2
4,0,0,0


In [12]:
print(counterfactual_df.describe())

       observed_sales  counterfactual_sales  estimated_uplift
count      5510.00000           5510.000000       5510.000000
mean          0.70127              0.484755          0.216515
std           2.78491              0.790695          2.869925
min           0.00000              0.000000        -10.000000
25%           0.00000              0.000000         -1.000000
50%           0.00000              0.000000          0.000000
75%           1.00000              1.000000          0.000000
max         122.00000             10.000000        121.000000


In [13]:
print(
    "Average Counterfactual Uplift:",
    counterfactual_df["estimated_uplift"].mean()
)

Average Counterfactual Uplift: 0.21651542649727767
